In [1]:
import random
import pandas as pd
from music21 import stream, tempo, metadata, instrument, roman, chord

In [3]:
import pandas as pd
from music21 import stream, chord, note, tempo, metadata, pitch, interval, key as m21key, roman
import re

ROMAN_BASES = {
    "I": "C", 
    "II": "D", 
    "III": "E",
    "IV": "F", 
    "V": "G", 
    "VI": "A", 
    "VII": "B"
}

""" ALPHABET
[1] "r"     "V+"    "ii"    "V"     "I"     "VI"    "iii"   "vi"    "IV"    "II"    "v"     "i"     "-VII" 
[14] "-III"  "-VI"   "-vii"  "vii"   "III"   "iv"    "-iiio" "VII"   "-iii"  "-II"   "#vo"   "#io"   "#iio" 
[27] "#ivo"  "vo"    "viio"  "-III+" "-viio" "iio"   "vio"   "-II+"  "-V"    "#I"    "#II"   "#IV"   "#V"   
[40] "iiio"  "I+"    "-iio"  "-vio"  "#iv"   "VII+"  "-ii"   "#IV+"  "-vi"   "io"    "III+"  "ivo"   "-I"   
[53] "-IV"   "-VII+" "#v"    "IV+"   "-i"    "-vo" 
"""

def roman_to_chord(symbol: str, key: str = "C"):
    """
    Convert a Roman numeral chord symbol to a music21.chord.Chord, 
    interpreted relative to the given key.
    
    Args:
        symbol (str): Roman numeral chord symbol (e.g. 'ii', 'V+', '-IIIo', etc.)
        key (str): Tonal center (e.g. 'C', 'G', 'Bb', 'F#', 'a', etc.)
    
    Returns:
        music21.chord.Chord: The constructed chord object.
    """
    # Extract modifiers (#, -) and core numeral (I–VII)
    match = re.match(r"([#\-]*)([ivIV]+)([+o]*)", symbol)
    if not match:
        return chord.Chord([])  # return an empty chord (rest) as fallback

    try:
        accidental_part, numeral, quality_part = match.groups()

        # Determine base root pitch
        numeral_upper = numeral.upper()
        if numeral_upper not in ROMAN_BASES:
            return chord.Chord([])
        # Get base degree in scale
        tonic_key = m21key.Key(key)
        base_pitch = pitch.Pitch(ROMAN_BASES[numeral_upper])
        # Transpose base_pitch to key context
        key_interval = interval.Interval(noteStart=pitch.Pitch("C"), noteEnd=tonic_key.tonic)
        root = key_interval.transposePitch(base_pitch)

        # Apply accidentals
        semitone_shift = accidental_part.count('#') - accidental_part.count('-')
        root.transpose(semitone_shift, inPlace=True)

        # Determine chord quality (major/minor/aug/dim)
        is_minor = numeral.islower()
        is_aug = '+' in quality_part
        is_dim = 'o' in quality_part

        # Construct chord tones via intervals
        third = interval.Interval('m3' if is_minor or is_dim else 'M3')
        fifth = interval.Interval('d5' if is_dim else ('A5' if is_aug else 'P5'))
        chord_pitches = [root, third.transposePitch(root), fifth.transposePitch(root)]

        return chord.Chord(chord_pitches)
    except Exception as e:
        print(f"Got an error when parsing {symbol}; fall back to parse as a rest: {e}")
        return chord.Chord([])  # return an empty chord (rest) as fallback


def round_value(val):
    try:
        return round(float(val), 2)
    except Exception:
        return val

def _filter_df_for_seq(df: pd.DataFrame, seq_num):
    s = str(seq_num)
    def keep(row):
        seq_val = str(row.get('Seq', ''))
        if seq_val in (s + 'Orig', s + 'Swap'):
            return True
        return False
    return df[df.apply(keep, axis=1)].copy()



In [4]:
TONICS = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

CHORD_INTERVAL_SEC = 2.4
BPM = 60.0 / CHORD_INTERVAL_SEC

### XML

In [5]:
df1 = pd.read_csv("./stimuli/sequences_1.csv")
df2 = pd.read_csv("./stimuli/sequences_2.csv")
df3 = pd.read_csv("./stimuli/sequences_3.csv")
df4 = pd.read_csv("./stimuli/sequences_4.csv")

dfs = {
    1: df1,
    2: df2,
    3: df3,
    4: df4
}

In [ ]:
def write_xml(
    df_seq: pd.DataFrame, 
    condition: int, 
    pair_num: int, 
    function: str, 
    tonic: str = "",
    out_dir='./stimuli_musicxml'
):  
    print(f"got tonic: {tonic}")
    if not tonic:
        # Pick a random tonic
        tonic = random.choice(TONICS)
        print(f"Randomly selected tonic: {tonic}")

    # Prepare scores/parts
    orig_score = stream.Score()
    orig_part = stream.Part()
    swap_score = stream.Score()
    swap_part = stream.Part()

    # Create a single MetronomeMark object
    mm = tempo.MetronomeMark(number=float(BPM))
    orig_part.insert(0.0, mm)
    swap_part.insert(0.0, mm)
    orig_score.insert(0.0, mm)
    swap_score.insert(0.0, mm)

    # metadata
    orig_score.metadata = metadata.Metadata(title=f"{condition}_pair{pair_num}_orig_{tonic}_{function}")
    swap_score.metadata = metadata.Metadata(title=f"{condition}_pair{pair_num}_swap_{tonic}_{function}")
    piano = instrument.Piano()
    orig_part.insert(0.0, piano)
    swap_part.insert(0.0, piano)

    # tonic
    tonic_chord = roman_to_chord("I", key=tonic)
    tonic_chord.quarterLength = 1
    rest = note.Rest()
    rest.quarterLength = 1
    orig_part.append(tonic_chord)
    orig_part.append(rest)
    swap_part.append(tonic_chord)
    swap_part.append(rest)

    # locate the pair in the given df
    df = _filter_df_for_seq(df_seq, pair_num)

    # generate MIDI / MusicXML
    for iter, row in df.iterrows():
        chord_obj = roman_to_chord(str(row['Symbol']), key=tonic)
        chord_obj.quarterLength = 1  # each chord = 1 quarter note -> with bpm=25 -> 2.4s apart
        ic, ent = round_value(row.get('IC', '')), round_value(row.get('Entropy', ''))
        lyric1 = f"  IC: {ic}; Ent: {ent}  "
        lyric2 = f"{row['Seq']}_{row['Row']}"
        chord_obj.addLyric(lyric1)
        chord_obj.addLyric(lyric2)
        seq_label = str(row['Seq'])[-4:]
        if seq_label == 'Orig':
            orig_part.append(chord_obj)
        else:
            swap_part.append(chord_obj)

    orig_score.append(orig_part)
    swap_score.append(swap_part)

    out_orig = f'{out_dir}/{condition}_pair{pair_num}_orig_{tonic}_{function}.musicxml'
    out_swap = f'{out_dir}/{condition}_pair{pair_num}_swap_{tonic}_{function}.musicxml'

    orig_score.write('musicxml', out_orig)
    swap_score.write('musicxml', out_swap)

    print("Wrote:", out_orig, "and", out_swap)
    return tonic, out_orig, out_swap



### WAV

In [7]:
import subprocess
import librosa
from scipy.io import wavfile
import numpy as np

def xml_to_wav(
        xml_file, wav_file, 
        click_wav="./snap.wav", click_gain=0.6,
        exe="/Applications/MuseScore 4.app/Contents/MacOS/mscore"):
    cmd = [exe, xml_file, "-o", wav_file]
    subprocess.run(cmd, check=True)

    audio_data, sr = librosa.load(wav_file)  # preserve original sample rate
    audio_data, _ = librosa.effects.trim(audio_data, top_db=40)  

    # add click sounds
    if click_wav:
        click_audio, click_sr = librosa.load(click_wav)
        if sr != click_sr:
            raise ValueError("Sample rates of click_audio and audio do not match!")
        quarter_duration = CHORD_INTERVAL_SEC
        sixteenth_duration = quarter_duration / 4
        sixteenth_samples = int(sixteenth_duration * sr)
        if len(click_audio) > sixteenth_samples:
            click_audio = click_audio[:sixteenth_samples]
        else:
            click_audio = np.pad(click_audio, (0, sixteenth_samples - len(click_audio)))
        click_audio = click_audio * click_gain

        tonic_len_samples = int(quarter_duration * sr) 
        rest_start = tonic_len_samples                 
        # Mix 4 clicks during rest only
        y_out = np.copy(audio_data)
        for i in range(4):
            start = rest_start + i * sixteenth_samples
            end = start + sixteenth_samples
            if end > len(y_out):
                break
            y_out[start:end] += click_audio

    wavfile.write(wav_file, sr, y_out)

    print("WAV written:", wav_file)
    return wav_file

In [24]:
stimuli_df = pd.read_csv("./stimuli.csv")

click_wav="./metro.wav"
click_gain=0.6

for iter, row in stimuli_df.iterrows():
    condition = int(row['condition'])
    tonic, xml_orig, xml_swap = write_xml(
        dfs[condition], 
        condition, 
        int(row['pair_num']),
        row['swap_function'],
        tonic = "" if pd.isna(row['tonic']) else row['tonic']
    )
    stimuli_df.loc[iter, 'tonic'] = tonic
    wav_orig = xml_to_wav(
        xml_orig, 
        "./stimuli_audio/" + xml_orig.split("/")[-1].replace(".musicxml", ".wav"),
        click_wav=click_wav, click_gain=click_gain
    )
    wav_swap = xml_to_wav(
        xml_swap, 
        "./stimuli_audio/" + xml_swap.split("/")[-1].replace(".musicxml", ".wav"),
        click_wav=click_wav, click_gain=click_gain
    )
    
stimuli_df.to_csv("./stimuli.csv", index=False)


got tonic: C
Wrote: ./stimuli_musicxml/1_pair21_orig_C_same.musicxml and ./stimuli_musicxml/1_pair21_swap_C_same.musicxml


qt.qml.typeregistration: Invalid QML element name "IconCode"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "MusicalSymbolCodes"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "ContainerType"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "NavigationEvent"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "MUAccessible"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "CompareType"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "SelectionMode"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "ToolBarItemType"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invali

WAV written: ./stimuli_audio/1_pair21_orig_C_same.wav


qt.qml.typeregistration: Invalid QML element name "IconCode"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "MusicalSymbolCodes"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "ContainerType"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "NavigationEvent"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "MUAccessible"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "CompareType"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "SelectionMode"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "ToolBarItemType"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invali

WAV written: ./stimuli_audio/1_pair21_swap_C_same.wav
got tonic: B
Wrote: ./stimuli_musicxml/1_pair25_orig_B_same.musicxml and ./stimuli_musicxml/1_pair25_swap_B_same.musicxml


...

WAV written: ./stimuli_audio/4_pair35_swap_C#_diff.wav
got tonic: C#
Wrote: ./stimuli_musicxml/4_pair34_orig_C#_diff.musicxml and ./stimuli_musicxml/4_pair34_swap_C#_diff.musicxml


qt.qml.typeregistration: Invalid QML element name "IconCode"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "MusicalSymbolCodes"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "ContainerType"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "NavigationEvent"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "MUAccessible"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "CompareType"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "SelectionMode"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "ToolBarItemType"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invali

WAV written: ./stimuli_audio/4_pair34_orig_C#_diff.wav


qt.qml.typeregistration: Invalid QML element name "IconCode"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "MusicalSymbolCodes"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "ContainerType"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "NavigationEvent"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "MUAccessible"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "CompareType"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "SelectionMode"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invalid QML element name "ToolBarItemType"; value type names should begin with a lowercase letter
qt.qml.typeregistration: Invali

WAV written: ./stimuli_audio/4_pair34_swap_C#_diff.wav
